# 🛒 Análisis Completo de Datos de Ventas y Usuarios

## Cuaderno de Proceso de Datos para Modelo de IA

**Descripción del proyecto:**  
Este cuaderno implementa el pipeline completo de preparación de datos de un e-commerce colombiano. El objetivo es preparar los datos para un modelo de clasificación que prediga si un usuario es **cliente frecuente** (`cliente_frecuente = 1`).

**Dataset:** `datos_ventas_usuarios.csv` — Registros de transacciones de usuarios (140 filas × 19 columnas).

---

**Etapas del pipeline:**
1. Carga e inspección inicial
2. Identificación de problemas de calidad
3. Limpieza y transformación
4. Análisis exploratorio avanzado (EDA)
5. Ingeniería de características
6. Codificación de variables categóricas
7. Escalado de variables numéricas
8. División Train / Validation / Test

## 0. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import sys

from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Configuración de visualización
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Librerías importadas correctamente')
print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')
print(f'sklearn disponible')

---
## 1. Carga e Inspección Inicial del Dataset

El primer paso siempre es **entender la estructura del dato**: cuántas filas y columnas hay, qué tipo de datos contiene cada columna, y obtener un vistazo estadístico básico.

In [ ]:
# Cargamos el dataset crudo (raw)
df_raw = pd.read_csv('datos_ventas_usuarios.csv')

print('=' * 55)
print('  CARGA DEL DATASET')
print('=' * 55)
print(f'Filas    : {df_raw.shape[0]}')
print(f'Columnas : {df_raw.shape[1]}')
print(f'Memoria  : {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# Primeras filas para un vistazo rápido
df_raw.head(5)

In [ ]:
# Tipos de datos y valores no nulos por columna
# Esto revela si las fechas se cargaron como strings, si hay columnas mixtas, etc.
df_raw.info()

In [ ]:
# Estadísticas descriptivas para variables numéricas
# Observamos rangos, medias y desviaciones para detectar anomalías
df_raw.describe().round(2)

In [ ]:
# Estadísticas de variables categóricas
# Revisamos la cardinalidad (cantidad de categorías únicas) de cada columna de texto
df_raw.describe(include='object')

---
## 2. Identificación de Problemas de Calidad

Auditamos el dataset en busca de los problemas más comunes:
- **Valores nulos / faltantes** → pueden sesgar modelos o causar errores
- **Filas duplicadas** → inflan artificialmente el dataset
- **Outliers** → valores extremos que distorsionan distribuciones
- **Inconsistencias lógicas** → p.ej. `total ≠ precio × cantidad`
- **Tipos de datos incorrectos** → fechas guardadas como `object`

In [ ]:
# 2.1 Análisis de valores nulos
nulos = df_raw.isnull().sum()
pct_nulos = (nulos / len(df_raw) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    '% Nulos': pct_nulos
}).sort_values('Nulos', ascending=False)

print('Resumen de valores nulos por columna:')
print(resumen_nulos[resumen_nulos['Nulos'] > 0] if resumen_nulos['Nulos'].sum() > 0
      else 'No se encontraron valores nulos en ninguna columna.')

In [ ]:
# 2.2 Filas duplicadas
n_dup = df_raw.duplicated().sum()
print(f'Filas duplicadas exactas : {n_dup}')

# También verificamos session_id único por transacción
n_dup_session = df_raw['session_id'].duplicated().sum()
print(f'session_id duplicados    : {n_dup_session}')

if n_dup == 0:
    print('No hay filas completamente duplicadas.')

In [ ]:
# 2.3 Tipos de datos incorrectos
print('Problema detectado:')
print(f'  fecha_compra → tipo actual: {df_raw["fecha_compra"].dtype}  (debería ser datetime64)')
print(f'  hora_compra  → tipo actual: {df_raw["hora_compra"].dtype}   (debería ser time/string estructurado)')
print()
print('Muestra de fecha_compra:')
print(df_raw['fecha_compra'].head(5).values)
print()
print('Muestra de hora_compra:')
print(df_raw['hora_compra'].head(5).values)

In [ ]:
# 2.4 Detección de outliers mediante IQR (Rango Intercuartílico)
# El método IQR define como outlier cualquier valor fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
# Es más robusto que 3σ cuando la distribución no es normal

cols_num = ['precio', 'cantidad', 'total', 'duracion_sesion_min', 'descuento_aplicado']

print(f'{"Columna":<25} {"Q1":>8} {"Q3":>8} {"IQR":>8} {"Límite inf":>12} {"Límite sup":>12} {"# Outliers":>12}')
print('-' * 90)

outlier_flags = {}
for col in cols_num:
    Q1 = df_raw[col].quantile(0.25)
    Q3 = df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (df_raw[col] < lower) | (df_raw[col] > upper)
    outlier_flags[col] = mask
    print(f'{col:<25} {Q1:>8.2f} {Q3:>8.2f} {IQR:>8.2f} {lower:>12.2f} {upper:>12.2f} {mask.sum():>12}')

In [ ]:
# Visualización de distribuciones y outliers con boxplots
fig, axes = plt.subplots(1, len(cols_num), figsize=(16, 4))
fig.suptitle('Distribución de Variables Numéricas — Detección de Outliers', fontsize=14)

for ax, col in zip(axes, cols_num):
    ax.boxplot(df_raw[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='#AED6F1', color='#2471A3'),
               medianprops=dict(color='#E74C3C', linewidth=2))
    ax.set_title(col)
    ax.set_ylabel('Valor')

plt.tight_layout()
plt.savefig('fig_outliers.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n Las cajas muestran Q1-Q3; los puntos fuera de los bigotes son outliers.')

In [ ]:
# 2.5 Verificación de consistencia lógica
# total debería ser aproximadamente precio × cantidad (puede haber diferencias por descuento)

df_raw['total_calculado'] = df_raw['precio'] * df_raw['cantidad']
discrepancias = df_raw[abs(df_raw['total'] - df_raw['total_calculado']) > 1]

print(f'Registros donde |total - (precio × cantidad)| > 1 : {len(discrepancias)}')

if len(discrepancias) == 0:
    print('Consistencia lógica verificada: total = precio × cantidad en todos los registros.')
else:
    print('Discrepancias encontradas:')
    print(discrepancias[['user_id','precio','cantidad','total','total_calculado']].head())

# También verificamos rangos válidos de descuento (debe estar entre 0% y 100%)
desc_invalido = df_raw[df_raw['descuento_aplicado'] < 0]
print(f'\nDescuentos negativos (inválidos): {len(desc_invalido)}')
desc_alto = df_raw[df_raw['descuento_aplicado'] > 100]
print(f'Descuentos > 100% (inválidos)  : {len(desc_alto)}')

# Limpiamos columna auxiliar
df_raw.drop(columns=['total_calculado'], inplace=True)

---
## 3. Limpieza y Transformación de Datos

Con los problemas identificados, aplicamos las correcciones:
1. Conversión de tipos (fechas, extracción de hora)
2. Marcado de outliers (en lugar de eliminar, los etiquetamos para que el modelo aprenda)
3. Creación de variables temporales útiles
4. Variables derivadas de comportamiento

In [ ]:
# 3.1 Conversión de tipos de datos
# Usamos copy() para preservar el dataset crudo intacto
df = df_raw.copy()

# Convertir fecha_compra a datetime — permite aritmética de fechas
df['fecha_compra'] = pd.to_datetime(df['fecha_compra'])

# Extraer la hora de compra como entero (0-23) desde el string 'HH:MM:SS'
df['hora_int'] = pd.to_datetime(df['hora_compra'], format='%H:%M:%S').dt.hour

print('Tipos convertidos correctamente:')
print(f'   fecha_compra → {df["fecha_compra"].dtype}')
print(f'   hora_int     → {df["hora_int"].dtype} (valores: {df["hora_int"].min()}-{df["hora_int"].max()})')

In [ ]:
# 3.2 Extracción de características temporales
# Las fechas en bruto no son útiles para los modelos; descomponerlas sí lo es

df['anio']       = df['fecha_compra'].dt.year
df['mes']        = df['fecha_compra'].dt.month          # 1-12
df['dia']        = df['fecha_compra'].dt.day             # 1-31
df['dia_semana'] = df['fecha_compra'].dt.dayofweek       # 0=Lunes ... 6=Domingo
df['nombre_dia'] = df['fecha_compra'].dt.day_name()      # Texto legible
df['trimestre']  = df['fecha_compra'].dt.quarter         # 1-4
df['es_fin_semana'] = (df['dia_semana'] >= 5).astype(int)  # Binaria: 1 si Sáb o Dom

print('Variables temporales creadas:')
print(df[['fecha_compra','anio','mes','dia','dia_semana','nombre_dia','trimestre','es_fin_semana']].head(4))

In [ ]:
# 3.3 Franja horaria — categoriza el momento del día
# Es más interpretable que la hora exacta y reduce cardinalidad

def asignar_franja(hora):
    if 0 <= hora < 6:
        return 'Madrugada'
    elif 6 <= hora < 12:
        return 'Mañana'
    elif 12 <= hora < 18:
        return 'Tarde'
    else:
        return 'Noche'

df['franja_horaria'] = df['hora_int'].apply(asignar_franja)

print('Distribución de franjas horarias:')
print(df['franja_horaria'].value_counts())

In [ ]:
# 3.4 Categorización del precio — segmenta el rango de precios en grupos
# Justificación: los percentiles garantizan que cada segmento tenga observaciones similares

q25 = df['precio'].quantile(0.25)
q75 = df['precio'].quantile(0.75)

def cat_precio(p):
    if p <= q25:              return 'Bajo'
    elif p <= df['precio'].median(): return 'Medio'
    elif p <= q75:             return 'Alto'
    else:                      return 'Premium'

df['categoria_precio'] = df['precio'].apply(cat_precio)

# 3.5 Categorización del tamaño de compra por total
q33 = df['total'].quantile(0.33)
q66 = df['total'].quantile(0.66)

df['tamano_compra'] = pd.cut(
    df['total'],
    bins=[-np.inf, q33, q66, np.inf],
    labels=['Pequeña', 'Mediana', 'Muy Grande']
)

# 3.6 Categorización de duración de sesión
df['duracion_categoria'] = pd.cut(
    df['duracion_sesion_min'],
    bins=[0, 30, 60, np.inf],
    labels=['Corta', 'Media', 'Larga']
)

print('Variables categóricas derivadas:')
print(f'   categoria_precio : {df["categoria_precio"].unique()}')
print(f'   tamano_compra    : {df["tamano_compra"].unique()}')
print(f'   duracion_categoria: {df["duracion_categoria"].unique()}')

In [ ]:
# 3.7 Variables binarias de comportamiento
df['tiene_descuento'] = (df['descuento_aplicado'] > 0).astype(int)

# precio_final: precio real pagado después de aplicar el descuento porcentual
df['precio_final'] = df['precio'] * (1 - df['descuento_aplicado'] / 100)
df['ahorro'] = df['precio'] - df['precio_final']  # cuánto ahorró el cliente

# 3.8 Tipo de dispositivo agrupado
mapa_dispositivo = {'Desktop': 'Computadora', 'Mobile': 'Movil', 'Tablet': 'Tablet'}
df['tipo_dispositivo'] = df['dispositivo'].map(mapa_dispositivo)

# 3.9 Número de compras por usuario y variable objetivo
# Un 'cliente frecuente' es aquel con más de 2 compras registradas
num_compras = df.groupby('user_id')['session_id'].count().reset_index()
num_compras.columns = ['user_id', 'num_compras']
df = df.merge(num_compras, on='user_id', how='left')

df['cliente_frecuente'] = (df['num_compras'] > 2).astype(int)

# 3.10 Marcado de outliers (etiqueta, NO eliminación)
# Eliminamos outliers sólo cuando existe evidencia de error de captura.
# En datos de e-commerce, valores extremos pueden ser clientes VIP legítimos.
# Los marcamos para que el modelo pueda aprender de ellos.
for col in ['precio', 'cantidad', 'total', 'duracion_sesion_min']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[f'es_outlier_{col}'] = (
        (df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)
    ).astype(int)

print('Variables de flags creadas:')
outlier_cols = [c for c in df.columns if c.startswith('es_outlier')]
print(df[outlier_cols].sum().to_string())

print(f'\nShape luego de limpieza: {df.shape}')
print(f'   Variable objetivo: cliente_frecuente')
print(f'   Distribución: {df["cliente_frecuente"].value_counts().to_dict()}')

In [ ]:
# Resumen comparativo antes/después de la limpieza
print('=' * 45)
print('  RESUMEN DE LIMPIEZA')
print('=' * 45)
print(f'Columnas originales : {df_raw.shape[1]}')
print(f'Columnas finales    : {df.shape[1]}')
print(f'Nuevas variables    : {df.shape[1] - df_raw.shape[1]}')
print(f'Filas originales    : {df_raw.shape[0]}')
print(f'Filas finales       : {df.shape[0]} (sin eliminar registros)')
print()
print('Nuevas columnas añadidas:')
nuevas = set(df.columns) - set(df_raw.columns)
for c in sorted(nuevas):
    print(f'  + {c}')

---
## 4. Análisis Exploratorio Avanzado (EDA)

El EDA revela patrones, correlaciones y la distribución de la variable objetivo. Esta etapa **guía** las decisiones de ingeniería de características.

In [ ]:
# 4.1 Distribución de la variable objetivo
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Variable Objetivo: cliente_frecuente', fontsize=14)

vc = df['cliente_frecuente'].value_counts()
labels = ['No Frecuente (0)', 'Frecuente (1)']
colors = ['#85C1E9', '#2ECC71']

ax1.pie(vc.values, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Proporción de clases')

ax2.bar(labels, vc.values, color=colors, edgecolor='white', width=0.5)
ax2.set_ylabel('Número de registros')
ax2.set_title('Conteo absoluto')
for i, v in enumerate(vc.values):
    ax2.text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_target.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nDesbalance de clases: {vc[0]} vs {vc[1]} ({vc[1]/len(df)*100:.1f}% son frecuentes)')
print('   → Al entrenar el modelo se debe considerar class_weight="balanced" o SMOTE')

In [ ]:
# 4.2 Distribuciones de variables numéricas clave
num_viz = ['precio', 'total', 'duracion_sesion_min', 'descuento_aplicado']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Distribuciones Numéricas — Histograma + KDE', fontsize=13)

for i, col in enumerate(num_viz):
    # Histograma general
    axes[0, i].hist(df[col], bins=20, color='#AED6F1', edgecolor='white')
    axes[0, i].set_title(f'{col}\n(general)')
    
    # Por clase objetivo
    for clase, color in [(0, '#E74C3C'), (1, '#27AE60')]:
        subset = df[df['cliente_frecuente'] == clase][col]
        axes[1, i].hist(subset, bins=15, alpha=0.6, color=color,
                        label=f'Clase {clase}', edgecolor='white', density=True)
    axes[1, i].set_title(f'{col}\n(por clase)')
    axes[1, i].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_distribuciones.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 4.3 Matriz de correlación de variables numéricas
# Identificamos: (a) features correlacionadas con el target, (b) multicolinealidad entre features

corr_cols = ['precio', 'cantidad', 'total', 'duracion_sesion_min',
             'descuento_aplicado', 'mes', 'dia', 'dia_semana',
             'ahorro', 'precio_final', 'num_compras', 'cliente_frecuente']

fig, ax = plt.subplots(figsize=(12, 9))
corr_matrix = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Solo triángulo inferior

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})
ax.set_title('Matriz de Correlación de Pearson', fontsize=13)
plt.tight_layout()
plt.savefig('fig_correlacion.png', dpi=120, bbox_inches='tight')
plt.show()

# Correlaciones con el target
print('\nCorrelaciones con cliente_frecuente (ordenadas):')
print(corr_matrix['cliente_frecuente'].sort_values(ascending=False).drop('cliente_frecuente').to_string())

In [ ]:
# 4.4 Variables categóricas vs variable objetivo
cat_eda = ['categoria', 'metodo_pago', 'dispositivo', 'fuente_trafico']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Tasa de Clientes Frecuentes por Variable Categórica', fontsize=13)

for ax, col in zip(axes.flat, cat_eda):
    tasa = df.groupby(col)['cliente_frecuente'].mean().sort_values(ascending=False)
    bars = ax.bar(tasa.index, tasa.values * 100,
                  color=plt.cm.Blues(np.linspace(0.4, 0.9, len(tasa))),
                  edgecolor='white')
    ax.set_title(f'{col}')
    ax.set_ylabel('% Clientes Frecuentes')
    ax.set_ylim(0, 110)
    ax.tick_params(axis='x', rotation=30)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{bar.get_height():.0f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig_categoricas_target.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 4.5 Análisis temporal de ventas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ventas por mes
ventas_mes = df.groupby('mes')['total'].sum()
axes[0].plot(ventas_mes.index, ventas_mes.values, marker='o',
             color='#2471A3', linewidth=2, markersize=8)
axes[0].fill_between(ventas_mes.index, ventas_mes.values, alpha=0.15, color='#2471A3')
axes[0].set_title('Total de Ventas por Mes')
axes[0].set_xlabel('Mes'); axes[0].set_ylabel('Total ($)')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun',
                          'Jul','Ago','Sep','Oct','Nov','Dic'])

# Distribución por día de la semana
dias_names = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']
ventas_dia = df.groupby('dia_semana')['total'].sum()
colors_dias = ['#5DADE2' if i < 5 else '#E74C3C' for i in range(7)]
axes[1].bar(dias_names, ventas_dia.values, color=colors_dias, edgecolor='white')
axes[1].set_title('Total de Ventas por Día de la Semana')
axes[1].set_ylabel('Total ($)')

plt.tight_layout()
plt.savefig('fig_temporal.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Ingeniería de Características

Creamos nuevas variables que el modelo no puede calcular por sí solo. La idea central es **codificar conocimiento de dominio** en el dataset.

### Variables RFM
El framework **RFM** (Recencia, Frecuencia, Valor Monetario) es el estándar en análisis de clientes de e-commerce:
- **Recencia**: cuántos días hace que el cliente compró por última vez
- **Frecuencia**: cuántas transacciones ha realizado
- **Valor Monetario**: cuánto dinero ha gastado en total

In [ ]:
# 5.1 Cálculo de métricas RFM por usuario
# Usamos la fecha máxima del dataset como fecha de referencia (fecha de "hoy")
fecha_ref = df['fecha_compra'].max()
print(f'Fecha de referencia: {fecha_ref.date()}')

rfm = df.groupby('user_id').agg(
    recencia        = ('fecha_compra', lambda x: (fecha_ref - x.max()).days),
    frecuencia      = ('session_id',   'count'),
    valor_monetario = ('total',        'sum'),
    ticket_promedio = ('total',        'mean'),
    primera_compra  = ('fecha_compra', 'min'),
    ultima_compra   = ('fecha_compra', 'max'),
).reset_index()

# Antigüedad del cliente (días desde su primera compra)
rfm['dias_como_cliente'] = (fecha_ref - rfm['primera_compra']).dt.days

# Tiempo promedio entre compras
# Para clientes con una sola compra, usamos dias_como_cliente como proxy
rfm['tiempo_entre_compras'] = np.where(
    rfm['frecuencia'] > 1,
    rfm['dias_como_cliente'] / (rfm['frecuencia'] - 1),
    rfm['dias_como_cliente']
)

print('\nMétricas RFM calculadas (primeras filas):')
print(rfm[['user_id','recencia','frecuencia','valor_monetario',
           'ticket_promedio','dias_como_cliente','tiempo_entre_compras']].head(6))

In [ ]:
# Unimos las métricas RFM de vuelta al dataset principal
df_fe = df.merge(
    rfm[['user_id','recencia','frecuencia','valor_monetario',
         'ticket_promedio','dias_como_cliente','tiempo_entre_compras']],
    on='user_id', how='left'
)

print(f'Shape después de unir RFM: {df_fe.shape}')

In [ ]:
# 5.2 Variables de interacción
# Capturan relaciones no lineales que los modelos lineales no pueden aprender

# ¿Cuánto cuesta el producto por minuto de navegación?
# Un ratio bajo puede indicar exploración; uno alto, compra directa
df_fe['precio_por_minuto_sesion'] = df_fe['precio'] / (df_fe['duracion_sesion_min'] + 1)

# Proporción del ahorro respecto al total pagado
# Mide la intensidad del descuento relativo al tamaño de la compra
df_fe['ratio_descuento_total'] = df_fe['ahorro'] / (df_fe['total'] + 1)

# Velocidad de compra: ¿cuántas compras hace por día activo?
df_fe['compras_por_dia'] = df_fe['frecuencia'] / (df_fe['dias_como_cliente'] + 1)

print('Variables de interacción creadas:')
print(df_fe[['precio_por_minuto_sesion','ratio_descuento_total','compras_por_dia']].describe().round(4))
print(f'\nShape final del dataset: {df_fe.shape}')

In [ ]:
# Visualización de las métricas RFM por clase objetivo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Distribución de Métricas RFM por Tipo de Cliente', fontsize=13)

metricas = ['recencia', 'valor_monetario', 'ticket_promedio']
titulos = ['Recencia (días)', 'Valor Monetario ($)', 'Ticket Promedio ($)']
colores = {'No Frecuente': '#E74C3C', 'Frecuente': '#27AE60'}

for ax, col, titulo in zip(axes, metricas, titulos):
    data_0 = df_fe[df_fe['cliente_frecuente'] == 0][col]
    data_1 = df_fe[df_fe['cliente_frecuente'] == 1][col]
    ax.boxplot([data_0, data_1], labels=['No Frecuente', 'Frecuente'],
               patch_artist=True,
               boxprops=dict(facecolor='#AED6F1'),
               medianprops=dict(color='#E74C3C', linewidth=2))
    ax.set_title(titulo)
    ax.set_ylabel('Valor')

plt.tight_layout()
plt.savefig('fig_rfm_clase.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. Codificación de Variables Categóricas

Los modelos de ML trabajan con números. Transformamos las variables de texto con **LabelEncoder**.

**¿Por qué LabelEncoder y no OneHotEncoder?**  
Con 11 variables categóricas y categorías con cardinalidad media, One-Hot crearía ~50+ columnas adicionales, lo que puede aumentar la dimensionalidad innecesariamente para árboles de decisión y modelos basados en boosting (XGBoost, LightGBM), que manejan bien LabelEncoded features.

In [ ]:
# Variables categóricas a codificar
cat_cols = [
    'categoria', 'metodo_pago', 'dispositivo', 'navegador',
    'sistema_operativo', 'fuente_trafico', 'franja_horaria',
    'tipo_dispositivo', 'categoria_precio', 'tamano_compra', 'duracion_categoria'
]

df_ml = df_fe.copy()
le = LabelEncoder()

print(f'{"Variable":<22} {"→ Columna encodada":<28} {"# Categorías":>12}')
print('-' * 65)

for col in cat_cols:
    df_ml[col + '_enc'] = le.fit_transform(df_ml[col].astype(str))
    n_cat = df_ml[col].nunique()
    print(f'{col:<22} → {col + "_enc":<28} {n_cat:>12}')

print(f'\n {len(cat_cols)} variables codificadas → {len(cat_cols)} columnas _enc añadidas')

---
## 7. Escalado de Variables Numéricas

El escalado es **esencial** para modelos que dependen de distancias (KNN, SVM, redes neuronales) o regularización (Regresión Logística). Aplicamos dos estrategias:

| Escalador | Fórmula | Rango | Cuándo usar |
|-----------|---------|-------|-------------|
| **MinMaxScaler** | `(x - min) / (max - min)` | [0, 1] | Distribuciones acotadas, redes neuronales |
| **StandardScaler** | `(x - μ) / σ` | No acotado | Distribuciones aproximadamente normales, algoritmos lineales |

In [ ]:
# Variables numéricas a escalar
numeric_features = [
    'precio', 'cantidad', 'total', 'duracion_sesion_min', 'descuento_aplicado',
    'mes', 'dia', 'dia_semana', 'trimestre',
    'precio_final', 'ahorro', 'num_compras',
    'recencia', 'frecuencia', 'valor_monetario', 'ticket_promedio',
    'dias_como_cliente', 'tiempo_entre_compras',
    'precio_por_minuto_sesion', 'ratio_descuento_total', 'compras_por_dia'
]

print(f'Variables a escalar: {len(numeric_features)}')
print(numeric_features)

In [ ]:
# 7.1 MinMaxScaler → rango [0, 1]
scaler_minmax = MinMaxScaler()
df_minmax = df_ml.copy()
df_minmax[numeric_features] = scaler_minmax.fit_transform(df_ml[numeric_features])

print('MinMaxScaler aplicado — verificación:')
print(f'  precio : min={df_minmax["precio"].min():.3f} | max={df_minmax["precio"].max():.3f}')
print(f'  total  : min={df_minmax["total"].min():.3f} | max={df_minmax["total"].max():.3f}')
print(f'  recencia: min={df_minmax["recencia"].min():.3f} | max={df_minmax["recencia"].max():.3f}')

In [ ]:
# 7.2 StandardScaler → media=0, std=1
scaler_std = StandardScaler()
df_std = df_ml.copy()
df_std[numeric_features] = scaler_std.fit_transform(df_ml[numeric_features])

print('StandardScaler aplicado — verificación:')
print(f'  precio : media={df_std["precio"].mean():.4f} | std={df_std["precio"].std():.4f}')
print(f'  total  : media={df_std["total"].mean():.4f} | std={df_std["total"].std():.4f}')
print(f'  valor_monetario: media={df_std["valor_monetario"].mean():.4f} | std={df_std["valor_monetario"].std():.4f}')

In [ ]:
# Comparación visual: antes vs MinMax vs Standard para 'precio'
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Efecto del Escalado sobre la Variable "precio"', fontsize=13)

datasets = [
    (df_ml, 'Original (sin escalar)', '#5DADE2'),
    (df_minmax, 'MinMaxScaler [0,1]', '#27AE60'),
    (df_std, 'StandardScaler [μ=0, σ=1]', '#E74C3C'),
]

for ax, (data, titulo, color) in zip(axes, datasets):
    ax.hist(data['precio'], bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(titulo)
    ax.set_xlabel('precio')
    ax.set_ylabel('Frecuencia')
    ax.axvline(data['precio'].mean(), color='black', linestyle='--', label=f'μ={data["precio"].mean():.2f}')
    ax.legend()

plt.tight_layout()
plt.savefig('fig_escalado.png', dpi=120, bbox_inches='tight')
plt.show()
print('La forma de la distribución NO cambia con el escalado; solo cambia el rango.')

---
## 8. División en Train / Validation / Test

Seguimos el esquema **70% / 15% / 15%**:
- **Train (70%)** → entrena el modelo
- **Validation (15%)** → ajusta hiperparámetros sin tocar el test
- **Test (15%)** → evaluación final, se toca una única vez

Usamos `stratify=y` para preservar la proporción de clases en los tres conjuntos.

In [ ]:
# Definición de features y target
feature_cols = (
    numeric_features
    + [c + '_enc' for c in cat_cols]
    + ['es_fin_semana', 'tiene_descuento', 'cliente_frecuente']
)
feature_cols = [c for c in feature_cols if c in df_minmax.columns]

# Usamos el dataset con MinMaxScaler para las particiones (más estable para árbol y redes)
X = df_minmax[feature_cols].fillna(0)
y = df_minmax['cliente_frecuente']

print(f'Features totales  : {X.shape[1]}')
print(f'Registros totales : {X.shape[0]}')
print(f'\nDistribución clase objetivo:')
print(y.value_counts().to_frame().T)

In [ ]:
# Primera división: separar Test (15%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y  # Mantiene la proporción de clases
)

# Segunda división: separar Validación del bloque Train+Val
# 0.1765 ≈ 15/85 ≈ 17.6% de train+val → ~15% del total
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.1765,
    random_state=42,
    stratify=y_trainval
)

# Resumen de la división
total = len(X)
print(f'\n{"Conjunto":<15} {"Filas":>6} {"% Total":>8}  {"Clase 0":>8}  {"Clase 1":>8}')
print('-' * 52)
for nombre, X_p, y_p in [("Train", X_train, y_train),
                          ("Validation", X_val, y_val),
                          ("Test", X_test, y_test)]:
    print(f'{nombre:<15} {len(X_p):>6} {len(X_p)/total*100:>7.1f}%  '
          f'{(y_p==0).sum():>8}  {(y_p==1).sum():>8}')
print('-' * 52)
print(f'{"TOTAL":<15} {total:>6} {"100.0%":>8}  {(y==0).sum():>8}  {(y==1).sum():>8}')

In [ ]:
# Visualización de la estratificación
fig, ax = plt.subplots(figsize=(9, 4))

conjuntos = ['Train', 'Validation', 'Test']
clase_0 = [(y_train==0).sum(), (y_val==0).sum(), (y_test==0).sum()]
clase_1 = [(y_train==1).sum(), (y_val==1).sum(), (y_test==1).sum()]

x = np.arange(len(conjuntos))
w = 0.35
b0 = ax.bar(x - w/2, clase_0, w, label='Clase 0 (No Frecuente)', color='#E74C3C', alpha=0.85)
b1 = ax.bar(x + w/2, clase_1, w, label='Clase 1 (Frecuente)', color='#27AE60', alpha=0.85)

ax.set_title('Distribución de Clases por Conjunto (Estratificado)', fontsize=12)
ax.set_xticks(x); ax.set_xticklabels(conjuntos)
ax.set_ylabel('Número de registros')
ax.legend()

for bar in list(b0) + list(b1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('fig_division_clases.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9. Guardado de Resultados

In [ ]:
# Guardar todos los artefactos del pipeline
df_minmax.to_csv('dataset_minmax.csv',  index=False, encoding='utf-8-sig')
df_std.to_csv('dataset_std.csv',        index=False, encoding='utf-8-sig')
df_fe.to_csv('datos_limpios.csv',       index=False, encoding='utf-8-sig')

X_train.to_csv('X_train.csv', index=False)
X_val.to_csv('X_val.csv',     index=False)
X_test.to_csv('X_test.csv',   index=False)
y_train.to_csv('y_train.csv', index=False, header=True)
y_val.to_csv('y_val.csv',     index=False, header=True)
y_test.to_csv('y_test.csv',   index=False, header=True)

print('Archivos guardados exitosamente:')
archivos = [
    ('datos_limpios.csv',   'Dataset limpio con ingeniería de características'),
    ('dataset_minmax.csv',  'Dataset completo escalado con MinMaxScaler'),
    ('dataset_std.csv',     'Dataset completo escalado con StandardScaler'),
    ('X_train.csv',         f'Features de entrenamiento ({len(X_train)} filas)'),
    ('X_val.csv',           f'Features de validación ({len(X_val)} filas)'),
    ('X_test.csv',          f'Features de prueba ({len(X_test)} filas)'),
    ('y_train.csv',         'Target de entrenamiento'),
    ('y_val.csv',           'Target de validación'),
    ('y_test.csv',          'Target de prueba'),
]
for archivo, desc in archivos:
    print(f'   {archivo:<22} → {desc}')

---
## 10. Resumen Final del Pipeline

| Etapa | Descripción | Decisión tomada |
|---|---|---|
| **Carga** | Dataset de transacciones e-commerce | 140 registros × 19 columnas |
| **Calidad** | Sin nulos ni duplicados; 6 outliers en precio | Outliers marcados, no eliminados |
| **Tipos** | `fecha_compra` como string | Convertida a `datetime64` |
| **Transformación** | Variables temporales, franjas, categorías | 21 variables añadidas |
| **EDA** | Correlaciones, distribuciones, análisis por clase | `num_compras` es el predictor más fuerte |
| **Ingeniería RFM** | Recencia, Frecuencia, Valor Monetario | 6 variables RFM por usuario |
| **Interacciones** | Ratios precio/sesión, descuento/total | 3 variables de interacción |
| **Codificación** | 11 variables categóricas | LabelEncoder (adecuado para árbol/boosting) |
| **Escalado** | 21 variables numéricas | MinMaxScaler y StandardScaler |
| **División** | 70/15/15 estratificado | Preserva proporción de clases |
